# Objective Validation Notebook

This notebook validates the required baseline objectives:
1. Project environment and dependencies are available
2. Pytest runs successfully
3. FastAPI server starts without errors
4. Baseline endpoints work: `/health`, `/echo`, and `/docs`

## 1) Notebook Setup
This section imports required libraries, resolves the repository root, and initializes shared variables used by all validation steps.

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path
from urllib import request
from urllib.error import URLError

def find_repo_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "pyproject.toml").exists():
        return cwd
    if (cwd.parent / "pyproject.toml").exists():
        return cwd.parent
    raise RuntimeError("Could not locate repo root with pyproject.toml")

ROOT = find_repo_root()
PORT = 8010
BASE_URL = f"http://127.0.0.1:{PORT}"
server_proc = None
results = {}

print(f"Repo root: {ROOT}")
print(f"Python executable: {sys.executable}")

Repo root: c:\Users\mlamar\Working Directory(Local)\Forge\m2\da-rag-project-2026_mlamar2019_2
Python executable: c:\Users\mlamar\Working Directory(Local)\Forge\m2\da-rag-project-2026_mlamar2019_2\.venv\Scripts\python.exe


## 2) Environment Tooling Check
This section verifies the local execution environment is ready by checking the active Python interpreter and the uv CLI version.

In [3]:
def run_cmd(cmd: list[str], cwd: Path = ROOT) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print("$", " ".join(cmd))
    print(proc.stdout)
    if proc.stderr.strip():
        print("stderr:")
        print(proc.stderr)
    return proc

# Objective 1: Verify environment tooling
py_v = run_cmd([sys.executable, "--version"])
uv_v = run_cmd(["uv", "--version"])

results["environment_ready"] = (py_v.returncode == 0 and uv_v.returncode == 0)
print("environment_ready:", results["environment_ready"])

$ c:\Users\mlamar\Working Directory(Local)\Forge\m2\da-rag-project-2026_mlamar2019_2\.venv\Scripts\python.exe --version
Python 3.14.3

$ uv --version
uv 0.10.11 (006b56b12 2026-03-16)

environment_ready: True


## 3) Pytest Validation
This section runs the baseline test suite using uv and records whether tests pass successfully.

In [4]:
# Objective 2: Run pytest
pytest_run = run_cmd(["uv", "run", "pytest", "tests/"])
results["pytest_passed"] = pytest_run.returncode == 0
print("pytest_passed:", results["pytest_passed"])

$ uv run pytest tests/
============================= test session starts =============================
platform win32 -- Python 3.14.3, pytest-9.0.1, pluggy-1.6.0
rootdir: c:\Users\mlamar\Working Directory(Local)\Forge\m2\da-rag-project-2026_mlamar2019_2
configfile: pyproject.toml
plugins: anyio-4.11.0
collected 2 items

tests\test_api.py ..                                                     [100%]

============================== 2 passed in 0.46s ==============================

pytest_passed: True


## 4) FastAPI Server Startup Check
This section starts the FastAPI app with uvicorn on a dedicated local port and waits until the health endpoint responds.

In [5]:
# Objective 3: Start FastAPI server
cmd = [
    sys.executable,
    "-m",
    "uvicorn",
    "main:app",
    "--app-dir",
    str(ROOT / "src"),
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
]
server_proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

ready = False
for _ in range(60):
    try:
        with request.urlopen(f"{BASE_URL}/health", timeout=1) as resp:
            if resp.status == 200:
                ready = True
                break
    except URLError:
        time.sleep(0.5)

results["server_started"] = ready and server_proc.poll() is None
print("server_started:", results["server_started"])
if not results["server_started"]:
    raise RuntimeError("Server failed to start on port 8010")

server_started: True


## 5) Endpoint Validation
This section performs live HTTP checks against the running API and verifies expected behavior for /health, /echo, and /docs.

In [6]:
# Objective 4: Validate baseline endpoints
with request.urlopen(f"{BASE_URL}/health", timeout=5) as health_resp:
    health_data = json.loads(health_resp.read().decode("utf-8"))

echo_req = request.Request(
    f"{BASE_URL}/echo",
    data=json.dumps({"message": "hello"}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with request.urlopen(echo_req, timeout=5) as echo_resp:
    echo_data = json.loads(echo_resp.read().decode("utf-8"))

with request.urlopen(f"{BASE_URL}/docs", timeout=5) as docs_resp:
    docs_status = docs_resp.status

results["health_ok"] = health_data == {"status": "ok"}
results["echo_ok"] = echo_data == {"message": "hello"}
results["docs_ok"] = docs_status == 200

print("health response:", health_data)
print("echo response:", echo_data)
print("docs status:", docs_status)

health response: {'status': 'ok'}
echo response: {'message': 'hello'}
docs status: 200


## 6) Server Cleanup
This section stops the temporary uvicorn process so the notebook does not leave a background server running after validation.

In [7]:
if server_proc is not None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_proc.kill()

print("Server process cleaned up.")

Server process cleaned up.


## 7) Final Objective Summary
This final section aggregates all checks and asserts that every required objective has been met.

In [8]:
required = [
    "environment_ready",
    "pytest_passed",
    "server_started",
    "health_ok",
    "echo_ok",
    "docs_ok",
]
all_passed = all(results.get(k, False) for k in required)

print("Validation results:")
for key in required:
    print(f"- {key}: {results.get(key)}")

print("\nALL OBJECTIVES MET:", all_passed)
if not all_passed:
    raise AssertionError("One or more required objectives failed")

Validation results:
- environment_ready: True
- pytest_passed: True
- server_started: True
- health_ok: True
- echo_ok: True
- docs_ok: True

ALL OBJECTIVES MET: True


<!-- re-push -->